# Preparing the Wildfire Data

The original wildfire dataset contains millions of records and many variables that are unnecessary for this dashboard. This notebook demonstrates how the raw data is inspected, cleaned, and reduced to create two smaller files:

- `wildfire_agg.csv`, containing aggregated wildfire totals for the line and bar charts.
- `wildfire_sample.csv`, containing a reproducible sample of individual fires for the scatter plot and map.

Creating these files reduces the size of the data stored in the GitHub repository and allows the dashboard to load more efficiently.

## Load and Inspect the Raw Data

The data comes from the [US Wildfire Records, 6th Edition](https://www.kaggle.com/datasets/behroozsohrabi/us-wildfire-records-6th-edition) dataset available on Kaggle. After downloading the CSV file, it is loaded into a pandas DataFrame. The structure and first few rows are inspected to understand the size of the dataset and the variables it contains.

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw_wildfire.csv", low_memory=False)

In [ ]:
df.info()

In [ ]:
pd.set_option('display.max_columns', None)
df.head()

## Select Relevant Variables

The raw dataset has 39 variables. This dashboard only needs 8: year, cause, fire size, location, state, and the two day-of-year fields used to calculate containment time. `NWCG_CAUSE_CLASSIFICATION` is renamed to `CAUSE` for convenience.

In [ ]:
cols_needed = [
    'FIRE_YEAR', 
    'NWCG_CAUSE_CLASSIFICATION', 
    'FIRE_SIZE',
    'LATITUDE', 
    'LONGITUDE', 
    'STATE',
    'DISCOVERY_DOY', 
    'CONT_DOY'
    ]

df = df[cols_needed].copy()
df = df.rename(columns={'NWCG_CAUSE_CLASSIFICATION' : 'CAUSE'})

## Explore the Selected Data

Before transforming the data, several characteristics are examined. This includes missing values, the available cause categories, the distribution of fire sizes, and the years represented in the dataset. These checks help identify the cleaning and preparation steps required for the dashboard.

### Check for Missing Values


`CONT_DOY` is missing for a large share of records, which prevents `DAYS_TO_CONTAIN` from being calculated for those rows. They aren't dropped here because the location, year, and cause data in those rows are still valid and usable by the map. They will be excluded later when the scatter plot is built.

In [ ]:
df.isna().sum()

### Examine Cause Categories

The unique cause classifications and their frequencies are reviewed before the labels are standardized.

In [ ]:
df['CAUSE'].value_counts()

### Examine Fire Size

Fire size is heavily right-skewed. The majority of the fires are small and there are a few massive ones. A log scale will be used wherever fire size is plotted to address this.

In [ ]:
df["FIRE_SIZE"].describe()

### Examine the Available Years

Confirms the years covered and checks for gaps or partial years.

In [ ]:
df['FIRE_YEAR'].value_counts().sort_index()

## Clean and Prepare the Data

### Standardize Cause Labels

The cause classifications are converted into shorter display labels that are easier to read in the dashboard controls and visualizations. The `Missing data/not specified/undetermined` classification is shortened to `Undetermined`, while the `Natural` and `Human` classifications are kept the same.

In [ ]:
cause_display_names = {
    'Missing data/not specified/undetermined': 'Undetermined',
    'Natural':                                 'Natural',
    'Human':                                   'Human'
}

df['CAUSE'] = df['CAUSE'].map(cause_display_names)

In [ ]:
# Verify the results of our change
df["CAUSE"].value_counts()

### Calculate Containment Time

The scatter plot compares wildfire size with the amount of time required to contain the fire. The raw dataset provides the day of year on which each fire was discovered and contained, so containment time is calculated by subtracting the discovery day from the containment day.

In [ ]:
df['DAYS_TO_CONTAIN'] = df['CONT_DOY'] - df['DISCOVERY_DOY']

In [ ]:
df["DAYS_TO_CONTAIN"].describe()

Note: A small number of fires show a negative `DAYS_TO_CONTAIN`, caused by discovery and containment dates spanning a calendar year boundary. These rows are filtered out later, in the dashboard notebook, where containment time is actually used

## Create the Aggregated Dataset

The line and bar charts display summary values rather than individual wildfire records. The data is therefore grouped by state, year, and cause. For each group, two values are calculated:

- `FIRE_COUNT`: the number of recorded fires.
- `TOTAL_ACRES`: the total number of acres burned.

This produces a substantially smaller dataset while preserving the exact totals required for the two summary visualizations.

In [ ]:
wildfire_agg = df.groupby(['STATE', 'FIRE_YEAR', 'CAUSE']).agg(
    fire_count=('FIRE_SIZE', 'size'),
    total_acres=('FIRE_SIZE', 'sum')
).reset_index()

In [ ]:
wildfire_agg.head()

In [ ]:
wildfire_agg.shape

## Create the Individual-Fire Sample

The point map and scatter plot require individual wildfire observations rather than aggregated totals. However, plotting every record would create a large file and reduce dashboard performance. A random sample of 25,000 individual fires is therefore selected.

Setting `random_state=42` makes the sample reproducible, meaning the same fires will be selected whenever the notebook is run.

In [ ]:
sample_columns = [
    "LATITUDE",
    "LONGITUDE",
    "FIRE_YEAR",
    "CAUSE",
    "FIRE_SIZE",
    "STATE",
    "DAYS_TO_CONTAIN"
]

wildfire_sample = df.sample(n=25000, random_state=42)[sample_columns].copy()

In [ ]:
wildfire_sample.head()

In [ ]:
wildfire_sample.shape

### Check Representativeness

Comparing cause proportions in the sample to the full dataset confirms the
sample fairly represents all three categories before it's used in the
dashboard.

In [ ]:
wildfire_sample['CAUSE'].value_counts(normalize=True) * 100

In [ ]:
df['CAUSE'].value_counts(normalize=True) * 100

## Export the Processed Files

The aggregated and sampled datasets are exported as CSV files. These processed files are stored in the GitHub repository and loaded directly by the dashboard, so users do not need to download or process the complete raw wildfire dataset to run the application.

In [ ]:
wildfire_agg.to_csv('data/wildfire_agg.csv', index=False)
wildfire_sample.to_csv('data/wildfire_sample.csv', index=False)